# Notebook 18 – Model Optimization for Deployment

Measure parameter count, serialized size, latency, batch inference, CPU/GPU, and dynamic quantization; introduce ONNX and model serialization.

**Learning loop:** Concept → explain in your own words → simple example → implement → interpret.

In [1]:
import io, time, copy
import numpy as np
import torch
from torch import nn
torch.manual_seed(42); np.random.seed(42)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:',torch.__version__,'| device:',device)

PyTorch: 2.14.0+cpu | device: cpu


## Concept and trade-offs
Parameter count is only a rough complexity measure. Runtime depends on operators, memory bandwidth, input size, batch size, and hardware. Batch inference can amortize overhead but uses more memory. CPU can suit small workloads; GPU helps when parallel work offsets transfer/launch overhead. Dynamic quantization reduces precision primarily for Linear layers on CPU. ONNX is a portable exchange format, not a guarantee of faster inference.

In [2]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader,TensorDataset
import torch.nn.functional as F
d=load_digits(); x=torch.tensor(d.images,dtype=torch.float32).unsqueeze(1)/16; y=torch.tensor(d.target,dtype=torch.long)
x=F.interpolate(x,(16,16),mode='bilinear',align_corners=False)
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=.25,random_state=42,stratify=y)
train_loader=DataLoader(TensorDataset(x_train,y_train),batch_size=128,shuffle=True)
test_loader=DataLoader(TensorDataset(x_test,y_test),batch_size=128)
class DeployCNN(nn.Module):
    def __init__(self):
        super().__init__(); self.features=nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d((2,2))); self.head=nn.Linear(128,10)
    def forward(self,x): return self.head(self.features(x).flatten(1))
model=DeployCNN().to(device); optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
for epoch in range(3):
    model.train()
    for xb,yb in train_loader:
        xb,yb=xb.to(device),yb.to(device); optimizer.zero_grad(); loss=nn.functional.cross_entropy(model(xb),yb); loss.backward(); optimizer.step()
model.eval()
@torch.no_grad()
def accuracy(net,loader):
    correct=total=0
    for xb,yb in loader:
        pred=net(xb.to(next(net.parameters()).device)).argmax(1).cpu(); correct+=(pred==yb).sum().item(); total+=len(yb)
    return correct/total
def state_size_mb(net):
    buffer=io.BytesIO(); torch.save(net.state_dict(),buffer); return buffer.getbuffer().nbytes/(1024**2)
print('test accuracy:',round(accuracy(model,test_loader),3),'parameters:',sum(p.numel() for p in model.parameters()),'state MiB:',round(state_size_mb(model),3))

test accuracy: 0.396 parameters: 6090 state MiB: 0.026


In [3]:
def latency_ms(net,batch,repeats=30):
    net.eval(); batch=batch.to(next(net.parameters()).device)
    with torch.no_grad():
        for _ in range(5): net(batch)
        if batch.is_cuda: torch.cuda.synchronize()
        start=time.perf_counter()
        for _ in range(repeats): net(batch)
        if batch.is_cuda: torch.cuda.synchronize()
    return (time.perf_counter()-start)*1000/repeats
model_cpu=copy.deepcopy(model).cpu()
for batch_size in (1,16,64):
    ms=latency_ms(model_cpu,x_test[:batch_size]); print(f'CPU batch={batch_size:2d}: {ms:.3f} ms/batch, {ms/batch_size:.3f} ms/image')
if torch.cuda.is_available():
    gpu_model=copy.deepcopy(model).to('cuda'); print('GPU batch 64 ms:',round(latency_ms(gpu_model,x_test[:64]),3))
else: print('GPU benchmark skipped: CUDA unavailable')
quantized=torch.ao.quantization.quantize_dynamic(copy.deepcopy(model_cpu),{nn.Linear},dtype=torch.qint8)
print('quantized accuracy:',round(accuracy(quantized,test_loader),3),'state MiB:',round(state_size_mb(quantized),3),'CPU batch64 ms:',round(latency_ms(quantized,x_test[:64]),3))

CPU batch= 1: 0.471 ms/batch, 0.471 ms/image
CPU batch=16: 1.523 ms/batch, 0.095 ms/image
CPU batch=64: 2.724 ms/batch, 0.043 ms/image
GPU benchmark skipped: CUDA unavailable
quantized accuracy: 0.393 state MiB: 0.023 CPU batch64 ms: 3.224


C:\Users\sibyr\AppData\Local\Temp\ipykernel_24292\2313269370.py:16: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized=torch.ao.quantization.quantize_dynamic(copy.deepcopy(model_cpu),{nn.Linear},dtype=torch.qint8)
C:\Users\sibyr\AppData\Roaming\Python\Python314\site-packages\torch\ao\nn\quantized\modules\utils.py:72: UserWarning: torch.quantize_per_tensor, torch.qua

In [4]:
checkpoint={'state_dict':model_cpu.state_dict(),'model_class':'DeployCNN','input_shape':[1,16,16]}
serialized=io.BytesIO(); torch.save(checkpoint,serialized)
print('serialized checkpoint bytes:',serialized.getbuffer().nbytes)
# Optional ONNX export: install onnx first, then set RUN_ONNX=True.
RUN_ONNX=False
if RUN_ONNX:
    torch.onnx.export(model_cpu,torch.zeros(1,1,16,16),'deploy_cnn.onnx',input_names=['image'],output_names=['logits'],dynamic_axes={'image':{0:'batch'},'logits':{0:'batch'}},opset_version=17,dynamo=False)
    print('Exported deploy_cnn.onnx')
else: print('ONNX export skipped; enable after installing onnx.')

serialized checkpoint bytes: 27285
ONNX export skipped; enable after installing onnx.


## Interpret accuracy, size, and speed
| Candidate | Accuracy | Size | Speed | Notes |
|---|---:|---:|---:|---|
| FP32 CNN | measured | measured | measured | reference |
| Dynamic quantized | measured | measured | measured | CPU-focused; check quality |
| GPU FP32 (if available) | same model | same | measured | batch size affects throughput |

Choose against real latency, memory, and minimum-quality requirements. Only Linear layers are quantized here, so a convolution-heavy model may not speed up. ONNX export changes the interchange format; benchmark its runtime separately on deployment hardware.